# 05b — Transformations in SQL (Self-training)

> **Self-training.** This is the SQL half of a pair. <a href="$./05a_transformations_pyspark">05a — Transformations in PySpark</a> runs the same 16 examples in PySpark, in the same order and on the same data, and each example returns the same rows.

Examples 1–11 are the SQL side of the PySpark ↔ SQL comparison slides in the deck. Examples 12–16 add tuning, subqueries, CTEs and writing a silver table. One Python cell creates the inline sample data as temp views. Every example is a `%sql` cell.

| Exam Domain | Weight |
|---|---|
| 3 — Data Transformation and Modeling | **22%** |

## Learning Objectives

After this module you will be able to:
- Filter and reshape rows and columns: select, filter, add, rename, drop, split
- Clean values: NULLs and fake missing values, conditional values, types, dates and strings
- Combine data with inner, left, multi-key and cross joins, union and union all
- Deduplicate with `row_number` and aggregate with count, exact and approximate distinct count, mean and summary
- Tune a query with `spark.sql.shuffle.partitions` and a join hint
- Write the cleaned result to a silver table

**Prerequisites:** `00_setup` finds your catalog. Example 16 writes one table, and the SQL function at the end is stored in your `silver` schema.

| Exam objective | Examples |
|---|---|
| 3.1 Clean bronze data (nulls, data types) and write silver tables | 3 NULL values, 4 Conditional logic, 5 Types, dates and strings, 16 Write the result to Silver |
| 3.2 Combine data: inner, left, broadcast, multi-key and cross joins, union, union all | 6 Joins, 7 Union, 13 Join hints |
| 3.3 Manipulate columns and rows: add, drop, split, rename, filter, explode arrays | 1 Select and filter, 2 Projection, 11 Arrays and JSON |
| 3.4 Deduplicate and aggregate: count, approximate count distinct, mean, summary | 8 Aggregations, 10 Deduplication |
| 3.5 Basic tuning parameters, re-measure | 12 Shuffle partitions, 13 Join hints |
| Not named in the objectives, common in exam code | 9 Window functions, 14 Subqueries, 15 CTEs |

## Setup

`00_setup` sets your catalog. Only example 16 and the SQL function at the end need it. The Python cell below creates the sample data and registers it as temp views. It is the same cell as in 05a, and it is the only Python cell in this notebook.

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Inline sample data for every example, registered as temp views (same cell in 05a and 05b)
customers = spark.createDataFrame([
    (1, "Anna", "Nowak", "anna.nowak@mail.com", "600100200", " Warsaw ", "2024-01-15", "Premium"),
    (2, "Piotr", "Kowalski", "piotr.k@shop.pl", None, "KRAKOW", "2024-02-03", "Standard"),
    (3, "Ewa", "Wisniewska", "ewa.w@mail.com", "N/A", "Gdansk", "2024-02-20", "Premium"),
    (4, "Jan", "Zielinski", "jan.z@shop.pl", "600300400", "warsaw", "2024-03-11", "Standard"),
], "customer_id INT, first_name STRING, last_name STRING, email STRING, "
   "phone STRING, city STRING, signup_date STRING, segment STRING")

orders = spark.createDataFrame([
    (101, 1, "P1", "2024-03-01", 2, 40.0, "completed"),
    (102, 2, "P2", "2024-03-02", 1, 250.0, "completed"),
    (103, 1, "P2", "2024-03-05", 1, 250.0, "cancelled"),
    (104, 3, "P1", "2024-03-06", 3, 60.0, "completed"),
    (105, 1, "P3", "2024-03-08", 1, 120.0, "completed"),
    (106, 9, "P1", "2024-03-09", 1, 20.0, "pending"),     # customer 9 is not in customers
], "order_id INT, customer_id INT, product_id STRING, order_date STRING, "
   "quantity INT, amount DOUBLE, status STRING")

orders_late = spark.createDataFrame([
    (106, 9, "P1", "2024-03-09", 1, 20.0, "pending"),     # order 106 sent again
    (107, 2, "P3", "2024-03-10", 1, 120.0, "completed"),
], orders.schema)

customer_updates = spark.createDataFrame([
    (1, "anna.nowak@mail.com", "2024-03-01"),
    (1, "anna.nowak@mail.com", "2024-03-01"),      # exact duplicate row
    (2, "piotr.k@shop.pl", "2024-03-01"),
    (2, "piotr.kowalski@shop.pl", "2024-03-05"),   # newer email for customer 2
    (3, "ewa.w@mail.com", "2024-03-02"),
], "customer_id INT, email STRING, updated_at STRING")

baskets = spark.createDataFrame([
    (101, ["pen", "notebook"], '{"device": {"os": "ios"}, "total": 40.0}'),
    (102, ["mug"], '{"device": {"os": "android"}, "total": 250.0}'),
    (103, [], '{"device": {"os": "windows"}, "total": 250.0}'),   # empty basket
], "order_id INT, items ARRAY<STRING>, payload STRING")

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
orders_late.createOrReplaceTempView("orders_late")
customer_updates.createOrReplaceTempView("customer_updates")
baskets.createOrReplaceTempView("baskets")

| Temp view (and DataFrame) | Rows | Used in |
|---|---|---|
| `customers` | 4: customer 2 has a `NULL` phone, customer 3 the string `'N/A'`, customer 4 has no orders | 2, 3, 5, 6, 13–16 |
| `orders` | 6: order 106 belongs to customer 9, who is not in `customers` | 1, 4, 6, 7, 8, 9, 12–15 |
| `orders_late` | 2: repeats order 106 and adds order 107 | 7 |
| `customer_updates` | 5: an exact duplicate (customer 1) and two versions of customer 2 | 6, 10 |
| `baskets` | 3: array `items` (order 103 is empty) and a JSON string `payload` | 11 |

## 1. Select and filter

Most transformations start by keeping only the rows and columns you need. `WHERE` keeps rows, and the `SELECT` list keeps columns. The DataFrame API and SQL are two front ends to one engine: Catalyst optimizes both into the same kind of physical plan.

In [ ]:
%sql
-- Completed orders, three columns
SELECT order_id, customer_id, amount
FROM orders
WHERE status = 'completed'

You see 4 rows: orders 101, 102, 104 and 105. `EXPLAIN` prints the plan of the same query, which is the first thing to read when a query is slow.

In [ ]:
%sql
-- The plan (compare it with explain() in 05a)
EXPLAIN
SELECT order_id, customer_id, amount
FROM orders
WHERE status = 'completed'

Read the plan from the bottom up. The same query in 05a prints the same plan, because the optimizer sees one query whichever API wrote it. On these few inline rows the optimizer may apply the filter while planning, so the plan can be a single `LocalTableScan`. On a table you see a scan, a `Filter` and a `Project`. The Photon note under the plan ("does not fully support ... LocalTableScan") only means that inline rows run outside Photon.

## 2. Projection: add, rename, drop

You reshape columns when the source layout does not fit the next step, for example to get the domain out of an email. `split` returns an array, and `[1]` takes its second element (indexes start at 0). `AS` renames a column, and `* EXCEPT (...)` selects every column except the listed ones. In PySpark the same steps are `withColumn`, `withColumnRenamed` and `drop`.

In [ ]:
%sql
-- Add email_domain, rename one column, drop two
SELECT * EXCEPT (phone, city, segment),
       segment AS tier,
       split(email, '@')[1] AS email_domain
FROM customers

You see 4 customers with 7 columns, ending with `tier` and `email_domain` (`mail.com` or `shop.pl`). On a value without `@` the array has one element, and `[1]` fails under ANSI mode (`INVALID_ARRAY_INDEX`). `try_element_at(split(email, '@'), 2)` returns `NULL` instead.

## 3. NULL values

A missing value is not always `NULL`. Customer 2 has a real `NULL` phone, and customer 3 has the string `'N/A'`. `IS NULL` finds only the first one, so look for such placeholder strings in raw data before you count or clean missing values.

In [ ]:
%sql
-- Real NULLs only: 'N/A' is a string, not NULL
SELECT * FROM customers WHERE phone IS NULL

One row: customer 2. To clean the column, first turn the fake value into `NULL` with `NULLIF`. Then replace every `NULL` with a default with `COALESCE`. `NVL` and `IFNULL` take two arguments and do the same.

In [ ]:
%sql
-- 'N/A' -> NULL with NULLIF, then a default
SELECT customer_id,
       COALESCE(NULLIF(phone, 'N/A'),
                'unknown') AS phone
FROM customers

Customers 2 and 3 now both show `unknown`, and customers 1 and 4 keep their numbers.

## 4. Conditional logic: when and CASE

Use `CASE WHEN` to turn a value into a category, such as a size band for a report. It checks the conditions in order, and the first true one wins. `ELSE` gives the value for every other row. Without `ELSE` those rows get `NULL`.

In [ ]:
%sql
-- Size band per order: the first true branch wins
SELECT order_id, amount,
       CASE WHEN amount >= 200 THEN 'large'
            WHEN amount >= 50 THEN 'medium'
            ELSE 'small'
       END AS size_band
FROM orders

Six orders: 102 and 103 are `large`, 104 and 105 are `medium`, 101 and 106 are `small`.

## 5. Types, dates and strings

Raw files often bring dates as strings and text with stray spaces or mixed case. Fix them before you group or join on them.
- `CAST(... AS DATE)` turns a `yyyy-MM-dd` string into a `DATE` (`to_date` does the same).
- `date_format` formats a date as text.
- `concat_ws` joins columns with a separator and skips `NULL`s.

In [ ]:
%sql
-- Text cleanup, a DATE column and a month label
SELECT customer_id,
       concat_ws(' ', first_name, last_name)
         AS full_name,
       lower(trim(city)) AS city,
       CAST(signup_date AS DATE) AS signed_up,
       date_format(CAST(signup_date AS DATE),
                   'yyyy-MM') AS signup_month
FROM customers

`' Warsaw '` and `'warsaw'` both become `warsaw`, `signed_up` has the type `DATE`, and `signup_month` is a string such as `2024-02`.

## 6. Joins

A join adds columns from a second table, for example the customer name to every order. `USING (customer_id)` keeps one `customer_id` column in the result. `ON o.customer_id = c.customer_id` keeps both key columns, so you have to qualify the one you select.

### Left join

Use a left join when every row of the left table must stay, even without a match.

In [ ]:
%sql
-- Every order, with the customer when one exists
SELECT order_id, customer_id, amount, first_name
FROM orders
LEFT JOIN customers USING (customer_id)

All 6 orders. Order 106 belongs to customer 9, who is not in `customers`, so its `first_name` is `NULL`.

### Inner join

Use an inner join when only rows with a match on both sides count.

In [ ]:
%sql
-- Only orders whose customer exists
SELECT order_id, customer_id, amount, first_name
FROM orders
JOIN customers USING (customer_id)

5 rows: the inner join (the default type) drops order 106. If the right table had a `customer_id` twice, every matching order would appear twice in both joins. Check that the key is unique on the side you join to.

### Two join keys

Join on two keys when one column alone does not identify the match.

In [ ]:
%sql
-- Two keys: updates that keep the current email
SELECT customer_id, email, updated_at, first_name
FROM customer_updates
JOIN customers USING (customer_id, email)

4 rows: both copies for customer 1, the old email of customer 2, and customer 3. The new email of customer 2 has no match. When the key names differ, write `ON a.k1 = b.x AND a.k2 = b.y`.

### Cross join

A cross join has no join key. It pairs every row on the left with every row on the right.

In [ ]:
%sql
-- Every product with every segment (3 x 2 rows)
SELECT product_id, segment
FROM (SELECT DISTINCT product_id
      FROM orders) p
CROSS JOIN (SELECT DISTINCT segment
            FROM customers) s

6 rows: a cross join returns rows(left) x rows(right). Use it on purpose, for example to list every product for every segment before a left join fills in the sales. Never run it by accident on two large tables.

## 7. Union

`orders_late` is a second batch of orders, as when new files arrive later. It repeats order 106 and adds order 107.

### UNION ALL

`UNION ALL` stacks two results by column **position** and keeps duplicates.

In [ ]:
%sql
-- Both batches stacked: duplicates stay
SELECT * FROM orders
UNION ALL
SELECT * FROM orders_late

8 rows, with order 106 twice. SQL set operators match columns by position, so list the columns in the same order on both sides. In PySpark, `unionByName` matches them by name.

### UNION

`UNION` (the same as `UNION DISTINCT`) also removes duplicate rows, which costs a shuffle.

In [ ]:
%sql
-- Duplicates removed: UNION is UNION DISTINCT
SELECT * FROM orders
UNION
SELECT * FROM orders_late

7 rows: the second copy of order 106 is gone.

## 8. Aggregations

An aggregation turns rows into figures per group, such as orders and customers per product.
- `GROUP BY` returns one row per group.
- `HAVING` filters the groups after the aggregation, and `WHERE` filters rows before it.
- `COUNT(DISTINCT)` is exact.
- `approx_count_distinct` uses a small HyperLogLog++ sketch with a default relative standard deviation of 5%, and it is much cheaper on large data.

In [ ]:
%sql
-- Per product: orders, customers, mean amount
SELECT product_id,
       COUNT(*) AS order_count,
       COUNT(DISTINCT customer_id)
         AS customer_count,
       approx_count_distinct(customer_id)
         AS customer_approx,
       AVG(amount) AS avg_amount
FROM orders
GROUP BY product_id
HAVING COUNT(*) >= 2

2 rows: `P1` (3 orders, 3 customers, mean 40.0) and `P2` (2 orders, 2 customers, mean 250.0). The result grid shows a `DOUBLE` without the trailing `.0`, so the means appear as `40` and `250`. `P3` has one order, so `HAVING` removes it. On data this small the approximate count equals the exact one. The DataFrame method `summary()` returns statistics of every column in one call. It has no single SQL equivalent, so in SQL you write the aggregates you need.

## 9. Window functions

Use a window function to number the orders of each customer or to compare a row with the one before it. It computes a value for every row from the rows of its partition and does not collapse them like `GROUP BY`. `OVER (PARTITION BY ... ORDER BY ...)` defines the window.

### row_number, lag and lead

`row_number` numbers the rows, and `lag` and `lead` read the previous and the next row.

In [ ]:
%sql
-- Order sequence and previous amount per customer
SELECT order_id, customer_id, order_date, amount,
       row_number() OVER (
         PARTITION BY customer_id
         ORDER BY order_date) AS order_seq,
       lag(amount) OVER (
         PARTITION BY customer_id
         ORDER BY order_date) AS prev_amt,
       lead(amount) OVER (
         PARTITION BY customer_id
         ORDER BY order_date) AS next_amt
FROM orders

Customer 1 has orders 101, 103 and 105 with `order_seq` 1, 2, 3, `prev_amt` `NULL`, 40.0, 250.0 and `next_amt` 250.0, 120.0, `NULL`. Every other customer has one order, so both are `NULL`.

### rank and dense_rank

`rank` and `dense_rank` differ from `row_number` only on ties. Orders 102 and 103 both have 250.0, so ranking all orders by amount shows the difference. A window without `PARTITION BY` moves all rows to one partition. That is fine here and slow on large data.

In [ ]:
%sql
-- Rank all orders by amount: ties share a rank
SELECT order_id, amount,
       rank() OVER (ORDER BY amount DESC) AS rnk,
       dense_rank() OVER (ORDER BY amount DESC)
         AS dense_rnk
FROM orders

Orders 102 and 103 share rank 1. The next order (105, 120.0) gets `rnk` 3, because `rank` leaves a gap after a tie, and `dense_rnk` 2, because `dense_rank` does not.

## 10. Deduplication

`customer_updates` has customer 1 twice with identical rows and customer 2 twice with different emails. When a source sends several versions of a record, you usually keep the **latest** row per key. Number the rows of each key newest first and keep number 1. `QUALIFY` filters on a window function without a subquery.

In [ ]:
%sql
-- Latest row per customer: QUALIFY keeps row 1
SELECT *
FROM customer_updates
QUALIFY row_number() OVER (
  PARTITION BY customer_id
  ORDER BY updated_at DESC) = 1

3 rows, one per customer. Customer 2 keeps `piotr.kowalski@shop.pl` from 2024-03-05. For contrast, `DISTINCT` removes only exact copies.

In [ ]:
%sql
-- Contrast: DISTINCT drops exact copies
SELECT DISTINCT * FROM customer_updates

4 rows: customer 1 once, customer 2 still twice. In PySpark, `dropDuplicates(["customer_id"])` would return 3 rows, but you cannot choose which row of customer 2 it keeps. When the latest row must win, use `row_number` with `QUALIFY`.

## 11. Arrays and JSON

JSON files and APIs often deliver lists and nested objects in one column. `baskets` has an array column `items` (order 103 has an empty array) and a JSON string `payload`.

### explode and explode_outer

`explode` turns each array element into its own row.

In [ ]:
%sql
-- One row per item: order 103 (empty) disappears
SELECT order_id, explode(items) AS item
FROM baskets

3 rows: two for order 101 and one for order 102. An empty or `NULL` array produces no row, so use the outer form when every order must stay.

In [ ]:
%sql
-- LATERAL VIEW OUTER keeps order 103 with NULL
SELECT order_id, item
FROM baskets
LATERAL VIEW OUTER explode(items) t AS item

4 rows: order 103 is back with `item` = `NULL`. `explode_outer(items)` in the `SELECT` list does the same.

### posexplode

`posexplode` also returns the position of each element in the array.

In [ ]:
%sql
-- posexplode: element and its 0-based position
SELECT order_id, posexplode(items) AS (pos, item)
FROM baskets

3 rows: `pen` at position 0 and `notebook` at 1 for order 101, and `mug` at 0 for order 102. `LATERAL VIEW OUTER posexplode(items) t AS pos, item` keeps order 103 with `NULL`s.

### from_json

Use `from_json` when you need typed columns from a JSON string. It parses the string with a schema (here a DDL string) into a typed struct. You then read nested fields with a dot.

In [ ]:
%sql
-- Parse the JSON string into a typed struct
WITH parsed AS (
  SELECT order_id, from_json(payload,
    'device STRUCT<os: STRING>, total DOUBLE')
    AS data
  FROM baskets)
SELECT order_id, data.device.os, data.total
FROM parsed

3 rows with `os` (`STRING`) and `total` (`DOUBLE`, typed by the schema).

### JSON path

The Databricks `:` path syntax is the quick way to read one value from a JSON string without a schema. It always returns a `STRING`, so cast it with `::`, for example `payload:total::DOUBLE`. In PySpark the same is `F.get_json_object("payload", "$.device.os")`.

In [ ]:
%sql
-- Path extraction without a schema: always STRING
SELECT order_id,
       payload:device.os AS os,
       payload:total AS total
FROM baskets

The same 3 values, but `total` is now the `STRING` `'40.0'`, while the `DOUBLE` of the previous result shows as `40`. Use `from_json` when you need types or many fields, and the path when you need one value.

## 12. Shuffle partitions

`spark.sql.shuffle.partitions` sets how many partitions a shuffle (`GROUP BY`, `JOIN`, `DISTINCT`) writes. The open-source default is 200. On serverless the value is `auto`, and AQE (Adaptive Query Execution) picks the number from the data. To test another value, change it for one session with `SET`, run the same query again, compare, and set it back.

In [ ]:
%sql
-- Read the current value
SET spark.sql.shuffle.partitions

In [ ]:
%sql
-- Set it for this session
SET spark.sql.shuffle.partitions = 8

In [ ]:
%sql
-- Run example 8 again with the new setting
SELECT product_id,
       COUNT(*) AS order_count,
       COUNT(DISTINCT customer_id)
         AS customer_count,
       approx_count_distinct(customer_id)
         AS customer_approx,
       AVG(amount) AS avg_amount
FROM orders
GROUP BY product_id
HAVING COUNT(*) >= 2

The same 2 rows as in example 8, because the setting changes only how the work is split. On large data, time both runs to see whether the new value helps. The next cell sets `auto`, the serverless value. On classic compute use the value the first cell showed (200 by default).

In [ ]:
%sql
-- Set the serverless value back
SET spark.sql.shuffle.partitions = auto

On serverless you can set only a short allow-list of Spark properties. It includes this key, `spark.sql.ansi.enabled`, `spark.sql.session.timeZone` and `spark.sql.files.maxPartitionBytes`. `spark.default.parallelism`, executor and driver memory and `spark.sql.autoBroadcastJoinThreshold` raise `CONFIG_NOT_AVAILABLE` there. On classic compute you set them in the cluster configuration. See [Spark properties on serverless](https://learn.microsoft.com/azure/databricks/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs).

## 13. Join hints

A broadcast join copies the small table to every task, so the large side is not shuffled. Spark broadcasts on its own below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default), and on these tiny tables it does so without a hint. A hint asks for a strategy explicitly. It works on every compute, including serverless, where the threshold cannot be set.

### Broadcast hint

The hint names the table or its alias exactly as written in `FROM`. A name that does not match is ignored.

In [ ]:
%sql
-- Inner join, small side broadcast to every task
SELECT /*+ BROADCAST(customers) */
       order_id, customer_id, segment
FROM orders
JOIN customers USING (customer_id)

5 rows, the same matches as the inner join in example 6. `EXPLAIN` on the same query shows which join strategy Spark chose.

In [ ]:
%sql
-- Look for BroadcastHashJoin in the plan
EXPLAIN
SELECT /*+ BROADCAST(customers) */
       order_id, customer_id, segment
FROM orders
JOIN customers USING (customer_id)

### Shuffle hash hint

`SHUFFLE_HASH` asks for a shuffled hash join instead, so you can compare the two plans.

In [ ]:
%sql
-- Same join with SHUFFLE_HASH: both sides shuffled
EXPLAIN
SELECT /*+ SHUFFLE_HASH(customers) */
       order_id, customer_id, segment
FROM orders
JOIN customers USING (customer_id)

The first plan shows `BroadcastHashJoin` (with Photon: `PhotonBroadcastHashJoin`) and a broadcast exchange on the customers side only. On serverless the plan shows it as `PhotonShuffleMapStage EXECUTOR_BROADCAST` with `PhotonShuffleExchangeSink SinglePartition` under it: the customers rows are collected into one partition and sent to every executor. The second plan shows `ShuffledHashJoin` (`PhotonShuffledHashJoin`) with a hash-partitioned exchange on both sides. The `16` in `hashpartitioning(customer_id#…, 16)` is not a setting: `spark.sql.shuffle.partitions` is `auto` (section 12), so AQE picked 16 partitions itself. `AdaptiveSparkPlan isFinalPlan=false` at the top means that AQE can still change the strategy at run time.

## 14. Subqueries

A subquery uses the result of one query inside another, for example to compare each order with the average. In PySpark, `IN` and `EXISTS` are a `left_semi` join, and `NOT EXISTS` is a `left_anti` join.

### Scalar subquery

A scalar subquery returns one value.

In [ ]:
%sql
-- Orders above the average amount
SELECT *
FROM orders
WHERE amount > (SELECT AVG(amount) FROM orders)

The average of the 6 orders is about 123.33, so you get 2 rows: orders 102 and 103 (250.0 each).

### IN subquery

`IN` tests a value against a one-column result.

In [ ]:
%sql
-- Customers with a cancelled order
SELECT *
FROM customers
WHERE customer_id IN (
  SELECT customer_id FROM orders
  WHERE status = 'cancelled')

1 row: customer 1 (order 103).

### NOT EXISTS

A correlated `EXISTS` runs per outer row. `NOT IN` returns no rows as soon as the subquery yields a `NULL`, and `NOT EXISTS` does not have this trap.

In [ ]:
%sql
-- Customers without any order
SELECT *
FROM customers c
WHERE NOT EXISTS (
  SELECT 1 FROM orders o
  WHERE o.customer_id = c.customer_id)

1 row: customer 4 (Jan) has no orders.

## 15. CTEs

A CTE gives a step of a long query a name, so you can read the query from top to bottom. `WITH name AS (...)` names a step that the main query (or a later CTE) reads. One query can define several CTEs, separated by commas. A CTE is only a definition: Spark plans the whole query at once, so it is not faster or slower than the same subquery.

In [ ]:
%sql
-- A named step (CTE): totals of completed orders
WITH completed_totals AS (
  SELECT customer_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY customer_id)
SELECT customer_id, first_name, total
FROM completed_totals
JOIN customers USING (customer_id)
ORDER BY total DESC

3 rows: Piotr 250.0, Anna 160.0 (orders 101 and 105) and Ewa 60.0 (the grid shows `250`, `160`, `60`). Order 106 is pending and belongs to an unknown customer, so it is not counted.

## 16. Write the result to Silver

A silver table keeps the cleaned data, so later steps do not repeat the cleaning. The cell combines examples 3–5: a real `NULL` instead of `'N/A'`, trimmed lower-case text and a `DATE` column. It saves the result as a Unity Catalog table in your `silver` schema with the module prefix `m05b_`. `CREATE OR REPLACE TABLE ... AS SELECT` replaces the table on every run.

In [ ]:
%sql
-- Clean customers (examples 3-5) and save to Silver
CREATE OR REPLACE TABLE silver.m05b_customers_clean AS
SELECT customer_id,
       concat_ws(' ', first_name, last_name) AS full_name,
       lower(trim(city)) AS city,
       CAST(signup_date AS DATE) AS signed_up,
       NULLIF(phone, 'N/A') AS phone,
       segment
FROM customers;

-- The CTAS itself returns no rows; count what it wrote
SELECT count(*) AS rows_written FROM silver.m05b_customers_clean

In [ ]:
%sql
-- Read the table back
SELECT * FROM silver.m05b_customers_clean

4 rows: customers 2 and 3 have `phone` = `NULL`, and `signed_up` is stored as `DATE`. The Cleanup cell at the end drops the table.

## Beyond the deck

The two tools below are not on the comparison slides, and 05a shows their PySpark forms. `QUALIFY` (example 10) and the `:` path (example 11) are Databricks SQL syntax with no DataFrame method of the same name.

### SQL UDF

A SQL UDF gives an expression a name, so you can reuse one business rule in many queries. It is stored in Unity Catalog, so other notebooks and users with access can call it. It runs in the engine like a built-in function, without the serialization cost of a Python UDF. This one goes to the `silver` schema of your catalog with the module prefix `m05b_`, and the last cell drops it.

In [ ]:
%sql
-- SQL UDF: the size band of example 4 as a function
CREATE OR REPLACE FUNCTION silver.m05b_size_band(amount DOUBLE)
RETURNS STRING
RETURN CASE WHEN amount >= 200 THEN 'large'
            WHEN amount >= 50 THEN 'medium'
            ELSE 'small'
       END

In [ ]:
%sql
-- Call it like a built-in function
SELECT order_id, amount,
       silver.m05b_size_band(amount) AS size_band
FROM orders

The same 6 rows as in example 4.

### Higher-order functions

`transform`, `filter` and `exists` work on each element of an array without `explode`, so the number of rows does not change. Use them when you change or test the items but keep one row per order. `x -> ...` is a lambda over one element.

In [ ]:
%sql
-- Change, filter and test array elements in place
SELECT order_id,
       transform(items, x -> upper(x)) AS upper_items,
       filter(items, x -> x LIKE 'n%') AS n_items,
       exists(items, x -> x = 'mug') AS has_mug
FROM baskets

3 rows, one per basket: order 101 has `n_items` = `[notebook]`, order 102 has `has_mug` = `true`, and order 103 keeps its empty array.

## Summary

| # | Example | PySpark | SQL |
|---|---|---|---|
| 1 | Select and filter | `spark.table().filter().select()`, `explain()` | `SELECT ... FROM ... WHERE`, `EXPLAIN` |
| 2 | Projection | `withColumn`, `withColumnRenamed`, `drop`, `F.split()[1]` | `split()[1] AS ...`, `* EXCEPT (...)` |
| 3 | NULL values | `isNull()`, `F.when()` without `otherwise`, `F.coalesce` | `IS NULL`, `NULLIF`, `COALESCE` |
| 4 | Conditional logic | `F.when().otherwise()` | `CASE WHEN ... ELSE ... END` |
| 5 | Types, dates and strings | `F.to_date`, `F.date_format`, `F.trim`, `F.lower`, `F.concat_ws` | `CAST(... AS DATE)` and the same functions |
| 6 | Joins | `join(df, "key", "left")`, inner by default, `join(df, ["k1", "k2"])`, `crossJoin()` | `LEFT JOIN ... USING (key)`, `JOIN ... USING (k1, k2)`, `CROSS JOIN` |
| 7 | Union | `union()`, `unionByName()`, `union().distinct()` | `UNION ALL`, `UNION` |
| 8 | Aggregations | `groupBy().agg()` with `F.count`, `F.count_distinct`, `F.approx_count_distinct`, `F.mean`; `summary()` | `GROUP BY`, `HAVING`, `COUNT(DISTINCT)`, `approx_count_distinct`, `AVG` |
| 9 | Window functions | `Window.partitionBy().orderBy()`, `F.row_number`, `F.lag`, `F.lead`, `F.rank`, `F.dense_rank` | `OVER (PARTITION BY ... ORDER BY ...)` with the same functions |
| 10 | Deduplication | `F.row_number()` + `filter("rn = 1")`; `dropDuplicates()` | `QUALIFY row_number() OVER (...) = 1`; `DISTINCT` |
| 11 | Arrays and JSON | `F.explode`, `F.explode_outer`, `F.posexplode`, `F.from_json`, `F.get_json_object` | `explode`, `LATERAL VIEW OUTER explode`, `posexplode`, `from_json`, `payload:path` |
| 12 | Shuffle partitions | `spark.conf.get()`, `spark.conf.set()` | `SET key`, `SET key = value` |
| 13 | Join hints | `F.broadcast(df)`, `df.hint("shuffle_hash")` | `/*+ BROADCAST(t) */`, `/*+ SHUFFLE_HASH(t) */` |
| 14 | Subqueries | value via `.first()[0]`, `left_semi`, `left_anti` | scalar subquery, `IN`, `NOT EXISTS` |
| 15 | CTEs | a DataFrame in a variable | `WITH name AS (...)` |
| 16 | Write the result to Silver | `df.write.mode("overwrite").saveAsTable()` | `CREATE OR REPLACE TABLE ... AS SELECT` |

> **Exam tip:** the DataFrame API and SQL run on the same engine and are equally fast. `QUALIFY`, `* EXCEPT` and the `:` path are Databricks SQL syntax. The temp views live only in this session. The stored objects are the table from example 16 and the SQL function, which the next cell drops.

## Cleanup

Drop the table of example 16 and the SQL function from your `silver` schema.

In [ ]:
%sql
-- Drop the module's table and SQL function
DROP TABLE IF EXISTS silver.m05b_customers_clean;
DROP FUNCTION IF EXISTS silver.m05b_size_band;
SELECT 'Dropped silver.m05b_customers_clean and silver.m05b_size_band' AS result

← <a href="$./05a_transformations_pyspark">05a — Transformations in PySpark</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./06_delta_optimization">06 — Delta Optimization →</a>

## Self-check (trainer): participants can skip this cell

In [ ]:
%sql
-- Self-check (trainer): the example queries return the rows the text gives (assert_true fails the cell otherwise)
SELECT
  assert_true((SELECT count(*) FROM orders WHERE status = 'completed') = 4, 'Select and filter: 4 completed orders'),
  assert_true((SELECT count(*) FROM orders LEFT JOIN customers USING (customer_id)) = 6, 'Joins: the left join keeps all 6 orders'),
  assert_true((SELECT count(*) FROM orders JOIN customers USING (customer_id)) = 5, 'Joins: the inner join drops order 106'),
  assert_true((SELECT count(*) FROM (SELECT * FROM orders UNION ALL SELECT * FROM orders_late)) = 8, 'Union: UNION ALL keeps the second copy of order 106'),
  assert_true((SELECT count(*) FROM (SELECT * FROM orders UNION SELECT * FROM orders_late)) = 7, 'Union: UNION removes it'),
  assert_true((SELECT count(*) FROM (SELECT product_id FROM orders GROUP BY product_id HAVING count(*) >= 2)) = 2, 'Aggregations: only P1 and P2 have 2+ orders'),
  assert_true((SELECT count(*) FROM (SELECT * FROM customer_updates
                                     QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1)) = 3,
              'Deduplication: one latest row per customer'),
  assert_true(abs((SELECT avg(amount) FROM orders) - 740 / 6) < 0.01, 'Subqueries: the mean amount is 740 / 6');

SELECT 'Self-check passed' AS result